# Baseline Evaluation: OneRec-1.7B on Contrastive Dataset v0

**Goal**: Given a user's history, compute the model's log-likelihood of generating
the chosen item vs the rejected item. Measure how often the model already
prefers the positive item (preference accuracy). A random model gives 50%.

**Format** (from benchmark_data):
- System: `你是一位视频推荐系统专家，擅长捕捉用户的兴趣演变。请根据历史序列推荐后续视频。`
- User: history SIDs as `<|sid_begin|><s_a_X><s_b_Y><s_c_Z><|sid_end|>...`
- Score = mean log P of the 3 SID tokens for each item

In [ ]:
# ── Config ──────────────────────────────────────────────
MODEL_NAME = "OpenOneRec/OneRec-1.7B"      # change to local path if already downloaded
N_SAMPLES  = 100                            # number of pairs to evaluate (set to None for all)
DEVICE     = "cuda"                         # "auto", "cpu", "cuda:0", "mps"
DATASET    = "../data/contrastive_dataset_v0/valid.parquet"

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

## 1. Load model and tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded on {model.device}")

## 2. Load dataset

In [ ]:
df = pd.read_parquet(DATASET)
if N_SAMPLES is not None:
    df = df.sample(n=min(N_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
print(f"Evaluating {len(df)} pairs")
df.head(2)

## 3. Formatting helpers

Convert SID triples to the `<|sid_begin|><s_a_X><s_b_Y><s_c_Z><|sid_end|>` format.

In [ ]:
SYSTEM_PROMPT = "你是一位视频推荐系统专家，擅长捕捉用户的兴趣演变。请根据历史序列推荐后续视频。"


def sid_to_str(sid):
    """Convert a single SID triple [a, b, c] to token string."""
    a, b, c = int(sid[0]), int(sid[1]), int(sid[2])
    return f"<|sid_begin|><s_a_{a}><s_b_{b}><s_c_{c}><|sid_end|>"


def sids_to_str(sids):
    """Convert a list of SID triples to concatenated token string."""
    return "".join(sid_to_str(s) for s in sids)


def build_prompt(hist_sids):
    """Build the chat prompt from history SIDs (without the target)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sids_to_str(hist_sids)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# Quick sanity check
row0 = df.iloc[0]
print("Chosen SID string:", sids_to_str(row0["chosen_sids"]))
print("Rejected SID string:", sids_to_str(row0["rejected_sids"]))
print()
# Show prompt tail (last 200 chars) to verify format
prompt = build_prompt(row0["hist_sids"])
print("Prompt tail:", prompt[-200:])

## 4. Scoring function

For a given prompt + completion, compute the mean log-probability of the
completion tokens. This tells us how likely the model thinks this item is
as the next recommendation given the history.

In [ ]:
@torch.no_grad()
def score_completion(prompt_text, completion_text):
    """
    Compute mean log P(completion | prompt).
    
    Returns the average log-probability over the completion tokens.
    """
    full_text = prompt_text + completion_text
    
    # Tokenize
    prompt_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(model.device)
    full_ids = tokenizer.encode(full_text, return_tensors="pt").to(model.device)
    
    # The completion tokens start after prompt
    n_prompt = prompt_ids.shape[1]
    n_full = full_ids.shape[1]
    n_completion = n_full - n_prompt
    
    if n_completion <= 0:
        return float("-inf")
    
    # Forward pass
    outputs = model(full_ids)
    logits = outputs.logits  # (1, seq_len, vocab_size)
    
    # Log probabilities of each completion token
    # logits[0, t-1, :] predicts token at position t
    log_probs = torch.log_softmax(logits[0], dim=-1)
    
    completion_log_probs = []
    for i in range(n_prompt, n_full):
        token_id = full_ids[0, i]
        lp = log_probs[i - 1, token_id].item()
        completion_log_probs.append(lp)
    
    return np.mean(completion_log_probs)

## 5. Run evaluation

In [ ]:
results = []

for idx in tqdm(range(len(df)), desc="Scoring pairs"):
    row = df.iloc[idx]
    
    prompt_text = build_prompt(row["hist_sids"])
    chosen_text = sids_to_str(row["chosen_sids"])
    rejected_text = sids_to_str(row["rejected_sids"])
    
    score_chosen = score_completion(prompt_text, chosen_text)
    score_rejected = score_completion(prompt_text, rejected_text)
    
    results.append({
        "uid": row["uid"],
        "score_chosen": score_chosen,
        "score_rejected": score_rejected,
        "chosen_wins": score_chosen > score_rejected,
    })

results_df = pd.DataFrame(results)
print(f"Done. {len(results_df)} pairs scored.")

## 6. Results

In [ ]:
pref_acc = results_df["chosen_wins"].mean()
mean_gap = (results_df["score_chosen"] - results_df["score_rejected"]).mean()

print(f"Preference accuracy : {pref_acc:.4f}  (random = 0.50)")
print(f"Mean score gap      : {mean_gap:.4f}")
print(f"Chosen wins         : {results_df['chosen_wins'].sum()} / {len(results_df)}")
print()
print(results_df[["score_chosen", "score_rejected"]].describe().round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score distributions
axes[0].hist(results_df["score_chosen"], bins=30, alpha=0.6, label="chosen", color="seagreen")
axes[0].hist(results_df["score_rejected"], bins=30, alpha=0.6, label="rejected", color="salmon")
axes[0].set_xlabel("mean log P (per token)")
axes[0].set_ylabel("count")
axes[0].set_title("Score distributions")
axes[0].legend()

# Gap distribution
gaps = results_df["score_chosen"] - results_df["score_rejected"]
axes[1].hist(gaps, bins=30, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="crimson", linestyle="--", label="tie")
axes[1].set_xlabel("score gap (chosen − rejected)")
axes[1].set_ylabel("count")
axes[1].set_title(f"Preference accuracy = {pref_acc:.2%}")
axes[1].legend()

plt.tight_layout()
plt.show()